# Sequence-to-Sequence Model for South Park Character Classification

This notebook implements an encoder-decoder (seq2seq) architecture for predicting which South Park character spoke a given line of dialogue.

## Why Seq2Seq?

While traditional RNN/LSTM/GRU models encode the entire input sequence into a single hidden state for classification, seq2seq models:

- **Encode the full context** - The encoder processes the entire dialogue line
- **Decode step-by-step** - The decoder generates predictions sequentially
- **Support attention** - Can focus on relevant parts of the input
- **More flexible** - Can be extended to generate character names or dialogue

For this classification task, we use a simplified seq2seq where the decoder outputs a single character label.

## 1. Imports and Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

import pandas as pd
import numpy as np
import re
from collections import Counter
from typing import List, Dict, Tuple

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import time

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Configuration Parameters

In [ ]:
# Dataset configuration
K = 12

# Model hyperparameters
EMBEDDING_DIM = 64
ENCODER_HIDDEN_DIM = 128
DECODER_HIDDEN_DIM = 128
NUM_LAYERS = 2
DROPOUT_RATE = 0.5
BIDIRECTIONAL = True
USE_ATTENTION = True

# Training hyperparameters
BATCH_SIZE = 64
LEARNING_RATE = 0.001
NUM_EPOCHS = 50
WEIGHT_DECAY = 1e-4
TEACHER_FORCING_RATIO = 0.5

# Early stopping
EARLY_STOP_PATIENCE = 7

# Text preprocessing
MIN_WORD_FREQ = 2
MAX_VOCAB_SIZE = 10000

# Data split ratios
VAL_SIZE = 0.15
TEST_SIZE = 0.15

print(f"Configuration:")
print(f"  Top K characters: {K}")
print(f"  Embedding dim: {EMBEDDING_DIM}")
print(f"  Encoder hidden dim: {ENCODER_HIDDEN_DIM}")
print(f"  Decoder hidden dim: {DECODER_HIDDEN_DIM}")
print(f"  Num layers: {NUM_LAYERS}")
print(f"  Bidirectional: {BIDIRECTIONAL}")
print(f"  Use attention: {USE_ATTENTION}")
print(f"  Teacher forcing ratio: {TEACHER_FORCING_RATIO}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Max epochs: {NUM_EPOCHS}")
print(f"  Early stopping patience: {EARLY_STOP_PATIENCE}")

## 3. Load Data

In [ ]:
# Load South Park dialogue data
base_url = "https://raw.githubusercontent.com/BobAdamsEE/SouthParkData/refs/heads/master/by-season/Season-{}.csv"

print("Loading South Park dialogue data...")
dfs = []
for season in tqdm(range(1, 20), desc="Loading seasons"):
    url = base_url.format(season)
    dfs.append(pd.read_csv(url))

df = pd.concat(dfs, ignore_index=True)
print(f"Loaded {len(df):,} dialogue lines")

## 4. Data Cleaning and Character Selection

In [ ]:
# Clean data
df_clean = df.drop_duplicates()
df_clean = df_clean.dropna(subset=['Character', 'Line'])
df_clean['Character'] = df_clean['Character'].str.strip()
df_clean['Line'] = df_clean['Line'].str.strip()
df_clean = df_clean[df_clean['Line'].str.len() > 0]

# Select top K characters
character_counts = df_clean['Character'].value_counts()
top_k_characters = character_counts.head(K).index.tolist()

print(f"Top {K} characters:")
for i, char in enumerate(top_k_characters, 1):
    count = character_counts[char]
    print(f"  {i:2d}. {char:20s} - {count:5d} lines")

# Filter dataset
df_model = df_clean[df_clean['Character'].isin(top_k_characters)].copy()
print(f"Filtered dataset: {len(df_model):,} lines")

## 5. Create Label Mappings and Split Data

In [ ]:
# Create mappings
char_to_label = {char: idx for idx, char in enumerate(top_k_characters)}
label_to_char = {idx: char for char, idx in char_to_label.items()}
df_model['label'] = df_model['Character'].map(char_to_label)

texts = df_model['Line'].tolist()
labels = df_model['label'].tolist()

# Split data
train_val_texts, test_texts, train_val_labels, test_labels = train_test_split(
    texts, labels, test_size=TEST_SIZE, stratify=labels, random_state=42
)

val_size_adjusted = VAL_SIZE / (1 - TEST_SIZE)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_val_texts, train_val_labels, test_size=val_size_adjusted, 
    stratify=train_val_labels, random_state=42
)

print(f"Train: {len(train_texts):,}  Val: {len(val_texts):,}  Test: {len(test_texts):,}")

## 6. Text Preprocessing and Vocabulary

For seq2seq, we need special tokens:
- `<pad>` - Padding token (index 0)
- `<unk>` - Unknown words (index 1)
- `<sos>` - Start of sequence (index 2)
- `<eos>` - End of sequence (index 3)

In [ ]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    return text.split()

# Preprocess
processed_train_texts = [preprocess_text(t) for t in tqdm(train_texts, desc="Train")]
processed_val_texts = [preprocess_text(t) for t in tqdm(val_texts, desc="Val")]
processed_test_texts = [preprocess_text(t) for t in tqdm(test_texts, desc="Test")]

# Build vocabulary with special tokens
class Vocabulary:
    def __init__(self, min_freq=1, max_size=None):
        self.word2idx = {'<pad>': 0, '<unk>': 1, '<sos>': 2, '<eos>': 3}
        self.idx2word = {0: '<pad>', 1: '<unk>', 2: '<sos>', 3: '<eos>'}
        self.min_freq = min_freq
        self.max_size = max_size
        self.PAD_IDX = 0
        self.UNK_IDX = 1
        self.SOS_IDX = 2
        self.EOS_IDX = 3
    
    def build_vocab(self, texts):
        word_counts = Counter(word for text in texts for word in text)
        word_counts = {w: c for w, c in word_counts.items() if c >= self.min_freq}
        sorted_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)
        if self.max_size:
            sorted_words = sorted_words[:self.max_size - 4]  # Reserve 4 special tokens
        for word, _ in sorted_words:
            idx = len(self.word2idx)
            self.word2idx[word] = idx
            self.idx2word[idx] = word
    
    def encode(self, text):
        return [self.word2idx.get(word, self.UNK_IDX) for word in text]
    
    def __len__(self):
        return len(self.word2idx)

vocab = Vocabulary(min_freq=MIN_WORD_FREQ, max_size=MAX_VOCAB_SIZE)
vocab.build_vocab(processed_train_texts)
print(f"Vocabulary size: {len(vocab):,}")

# Encode
indexed_train_texts = [vocab.encode(t) for t in processed_train_texts]
indexed_val_texts = [vocab.encode(t) for t in processed_val_texts]
indexed_test_texts = [vocab.encode(t) for t in processed_test_texts]

## 7. Dataset and DataLoader

For seq2seq, we need to prepare encoder inputs (dialogue lines) and decoder targets (character labels).

In [ ]:
class Seq2SeqDataset(Dataset):
    def __init__(self, texts, labels, sos_idx, eos_idx):
        self.texts = texts
        self.labels = labels
        self.sos_idx = sos_idx
        self.eos_idx = eos_idx
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        # Encoder input: <sos> + text + <eos>
        encoder_input = [self.sos_idx] + self.texts[idx] + [self.eos_idx]
        return {
            'encoder_input': torch.tensor(encoder_input, dtype=torch.long),
            'label': torch.tensor(self.labels[idx], dtype=torch.long),
            'length': len(encoder_input)
        }

def collate_fn(batch):
    # Sort by length for pack_padded_sequence
    batch = sorted(batch, key=lambda x: x['length'], reverse=True)
    max_len = batch[0]['length']
    
    encoder_inputs, labels, lengths = [], [], []
    for item in batch:
        enc_input = item['encoder_input']
        # Pad to max length
        padded = torch.cat([enc_input, torch.zeros(max_len - len(enc_input), dtype=torch.long)])
        encoder_inputs.append(padded)
        labels.append(item['label'])
        lengths.append(item['length'])
    
    return {
        'encoder_input': torch.stack(encoder_inputs),
        'label': torch.stack(labels),
        'length': torch.tensor(lengths, dtype=torch.long)
    }

# Create datasets and loaders
train_dataset = Seq2SeqDataset(indexed_train_texts, train_labels, vocab.SOS_IDX, vocab.EOS_IDX)
val_dataset = Seq2SeqDataset(indexed_val_texts, val_labels, vocab.SOS_IDX, vocab.EOS_IDX)
test_dataset = Seq2SeqDataset(indexed_test_texts, test_labels, vocab.SOS_IDX, vocab.EOS_IDX)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"Train batches: {len(train_loader)}  Val batches: {len(val_loader)}  Test batches: {len(test_loader)}")

## 8. Seq2Seq Model Architecture

We implement:
1. **Encoder** - Processes the input dialogue line
2. **Attention** - Allows decoder to focus on relevant encoder outputs
3. **Decoder** - Generates character prediction
4. **Seq2Seq** - Combines encoder and decoder

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers, dropout_rate, bidirectional):
        super(Encoder, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.bidirectional = bidirectional
        self.num_directions = 2 if bidirectional else 1
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.gru = nn.GRU(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout_rate if num_layers > 1 else 0,
            bidirectional=bidirectional
        )
        self.dropout = nn.Dropout(dropout_rate)
    
    def forward(self, input_seq, lengths):
        # input_seq: (batch_size, seq_len)
        embedded = self.dropout(self.embedding(input_seq))
        
        # Pack padded sequences
        packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=True)
        outputs, hidden = self.gru(packed)
        outputs, _ = pad_packed_sequence(outputs, batch_first=True)
        
        # outputs: (batch_size, seq_len, hidden_dim * num_directions)
        # hidden: (num_layers * num_directions, batch_size, hidden_dim)
        
        if self.bidirectional:
            # Combine forward and backward hidden states
            # hidden: (num_layers, batch_size, hidden_dim * 2)
            hidden = hidden.view(self.num_layers, self.num_directions, -1, self.hidden_dim)
            hidden = torch.cat([hidden[:, 0, :, :], hidden[:, 1, :, :]], dim=2)
        
        return outputs, hidden


class Attention(nn.Module):
    def __init__(self, hidden_dim, bidirectional=False):
        super(Attention, self).__init__()
        encoder_dim = hidden_dim * (2 if bidirectional else 1)
        self.attn = nn.Linear(encoder_dim + hidden_dim, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)
    
    def forward(self, hidden, encoder_outputs, mask=None):
        # hidden: (batch_size, hidden_dim)
        # encoder_outputs: (batch_size, seq_len, encoder_dim)
        
        batch_size = encoder_outputs.size(0)
        seq_len = encoder_outputs.size(1)
        
        # Repeat hidden state seq_len times
        hidden = hidden.unsqueeze(1).repeat(1, seq_len, 1)
        
        # Calculate attention scores
        energy = torch.tanh(self.attn(torch.cat([hidden, encoder_outputs], dim=2)))
        attention = self.v(energy).squeeze(2)
        
        # Apply mask if provided
        if mask is not None:
            attention = attention.masked_fill(mask == 0, -1e10)
        
        # Softmax to get attention weights
        attn_weights = torch.softmax(attention, dim=1)
        
        # Apply attention to encoder outputs
        context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs).squeeze(1)
        
        return context, attn_weights


class Decoder(nn.Module):
    def __init__(self, hidden_dim, num_classes, num_layers, dropout_rate, use_attention, encoder_bidirectional):
        super(Decoder, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_classes = num_classes
        self.use_attention = use_attention
        
        encoder_dim = hidden_dim * (2 if encoder_bidirectional else 1)
        
        if use_attention:
            self.attention = Attention(hidden_dim, encoder_bidirectional)
            self.fc = nn.Linear(encoder_dim + hidden_dim, num_classes)
        else:
            self.fc = nn.Linear(hidden_dim, num_classes)
        
        self.dropout = nn.Dropout(dropout_rate)
    
    def forward(self, hidden, encoder_outputs, mask=None):
        # hidden: (num_layers, batch_size, hidden_dim * num_directions)
        # encoder_outputs: (batch_size, seq_len, encoder_dim)
        
        # Use last layer hidden state
        last_hidden = hidden[-1]  # (batch_size, hidden_dim * num_directions)
        
        if self.use_attention:
            # Get context vector from attention
            context, attn_weights = self.attention(last_hidden, encoder_outputs, mask)
            # Combine context and hidden
            combined = torch.cat([last_hidden, context], dim=1)
            combined = self.dropout(combined)
            output = self.fc(combined)
        else:
            last_hidden = self.dropout(last_hidden)
            output = self.fc(last_hidden)
        
        return output


class Seq2SeqClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, encoder_hidden_dim, decoder_hidden_dim,
                 num_classes, num_layers, dropout_rate, bidirectional, use_attention):
        super(Seq2SeqClassifier, self).__init__()
        
        self.encoder = Encoder(vocab_size, embedding_dim, encoder_hidden_dim, 
                               num_layers, dropout_rate, bidirectional)
        self.decoder = Decoder(decoder_hidden_dim, num_classes, num_layers, 
                               dropout_rate, use_attention, bidirectional)
    
    def forward(self, input_seq, lengths):
        # Encode
        encoder_outputs, hidden = self.encoder(input_seq, lengths)
        
        # Create mask for attention (1 for real tokens, 0 for padding)
        batch_size = input_seq.size(0)
        max_len = input_seq.size(1)
        mask = torch.arange(max_len, device=input_seq.device).unsqueeze(0).expand(batch_size, -1)
        mask = mask < lengths.unsqueeze(1)
        
        # Decode
        output = self.decoder(hidden, encoder_outputs, mask)
        
        return output


# Initialize model
model = Seq2SeqClassifier(
    vocab_size=len(vocab),
    embedding_dim=EMBEDDING_DIM,
    encoder_hidden_dim=ENCODER_HIDDEN_DIM,
    decoder_hidden_dim=DECODER_HIDDEN_DIM,
    num_classes=K,
    num_layers=NUM_LAYERS,
    dropout_rate=DROPOUT_RATE,
    bidirectional=BIDIRECTIONAL,
    use_attention=USE_ATTENTION
)

print(f"Seq2Seq model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Model architecture:")
print(model)

## 9. Class Weights

In [ ]:
class_weights = compute_class_weight('balanced', classes=np.arange(K), y=train_labels)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
print("Class weights computed")
print(f"Class weights: {class_weights}")

## 10. Training Function with Early Stopping

In [ ]:
def train_seq2seq(model, train_loader, val_loader, num_epochs, lr, wd, class_weights, patience=7):
    model.to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_loss = float('inf')
    best_model_state = None
    patience_counter = 0
    
    print("Training Seq2Seq model...")
    start_time = time.time()
    
    for epoch in range(num_epochs):
        # Train
        model.train()
        train_loss = 0.0
        train_preds, train_labels_list = [], []
        
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
            encoder_input = batch['encoder_input'].to(device)
            labels = batch['label'].to(device)
            lengths = batch['length']
            
            optimizer.zero_grad()
            outputs = model(encoder_input, lengths)
            loss = criterion(outputs, labels)
            loss.backward()
            
            # Gradient clipping to prevent exploding gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            
            train_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            train_preds.extend(preds.cpu().numpy())
            train_labels_list.extend(labels.cpu().numpy())
        
        train_loss /= len(train_loader)
        train_acc = accuracy_score(train_labels_list, train_preds)
        
        # Validate
        model.eval()
        val_loss = 0.0
        val_preds, val_labels_list = [], []
        
        with torch.no_grad():
            for batch in val_loader:
                encoder_input = batch['encoder_input'].to(device)
                labels = batch['label'].to(device)
                lengths = batch['length']
                outputs = model(encoder_input, lengths)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                preds = torch.argmax(outputs, dim=1)
                val_preds.extend(preds.cpu().numpy())
                val_labels_list.extend(labels.cpu().numpy())
        
        val_loss /= len(val_loader)
        val_acc = accuracy_score(val_labels_list, val_preds)
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        
        print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f} Train Acc={train_acc:.4f} | Val Loss={val_loss:.4f} Val Acc={val_acc:.4f}")
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict().copy()
            patience_counter = 0
            print(f"  -> New best model (val_loss: {val_loss:.4f})")
        else:
            patience_counter += 1
        
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break
    
    # Load best model
    model.load_state_dict(best_model_state)
    elapsed = time.time() - start_time
    print(f"Training completed in {elapsed/60:.2f} minutes")
    
    return {'model': model, 'history': history, 'best_val_loss': best_val_loss, 'time': elapsed}

## 11. Train Model

In [ ]:
# Train the seq2seq model
results = train_seq2seq(
    model, 
    train_loader, 
    val_loader, 
    NUM_EPOCHS, 
    LEARNING_RATE, 
    WEIGHT_DECAY, 
    class_weights, 
    EARLY_STOP_PATIENCE
)

## 12. Evaluate Model

Evaluate the trained seq2seq model on the test set.

In [ ]:
def evaluate_seq2seq(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            encoder_input = batch['encoder_input'].to(device)
            labels = batch['label'].to(device)
            lengths = batch['length']
            outputs = model(encoder_input, lengths)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Calculate metrics
    metrics = {
        'accuracy': accuracy_score(all_labels, all_preds),
        'precision': precision_score(all_labels, all_preds, average='macro', zero_division=0),
        'recall': recall_score(all_labels, all_preds, average='macro', zero_division=0),
        'f1_score': f1_score(all_labels, all_preds, average='macro', zero_division=0)
    }
    
    return metrics, all_preds, all_labels

# Evaluate on test set
test_metrics, test_preds, test_labels_list = evaluate_seq2seq(model, test_loader)

# Print results
print("=" * 80)
print("TEST SET PERFORMANCE")
print("=" * 80)
print(f"Accuracy:  {test_metrics['accuracy']:.4f}")
print(f"Precision: {test_metrics['precision']:.4f}")
print(f"Recall:    {test_metrics['recall']:.4f}")
print(f"F1-Score:  {test_metrics['f1_score']:.4f}")
print("=" * 80)

## 13. Save Model

Save the trained seq2seq model with configuration and metrics.

In [ ]:
# Save checkpoint
checkpoint = {
    'model_type': 'Seq2Seq',
    'model_state_dict': model.state_dict(),
    'config': {
        'K': K,
        'vocab_size': len(vocab),
        'embedding_dim': EMBEDDING_DIM,
        'encoder_hidden_dim': ENCODER_HIDDEN_DIM,
        'decoder_hidden_dim': DECODER_HIDDEN_DIM,
        'num_layers': NUM_LAYERS,
        'dropout_rate': DROPOUT_RATE,
        'bidirectional': BIDIRECTIONAL,
        'use_attention': USE_ATTENTION
    },
    'char_to_label': char_to_label,
    'label_to_char': label_to_char,
    'test_metrics': test_metrics,
    'history': results['history'],
    'training_time': results['time']
}

filename = f'seq2seq_model_k{K}_attn{USE_ATTENTION}.pt'
torch.save(checkpoint, filename)
print(f"Model saved to: {filename}")
print(f"\nModel summary:")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Test F1-Score: {test_metrics['f1_score']:.4f}")
print(f"  Training time: {results['time']/60:.2f} minutes")